In [ ]:
import os
from pathlib import Path
from src.spark_session import create_spark_session

local_data = os.getenv("FASTFOOD_LOCAL_DATA")
orders_path = (Path(local_data).resolve() / "orders_history.jsonl").as_uri() if local_data else "s3a://lake/bronze/history/orders_history.jsonl"
restaurants_path = (Path(local_data).resolve() / "restaurants.json").as_uri() if local_data else "s3a://lake/bronze/reference/restaurants.json"

from pyspark.sql import functions as F
from pyspark.sql import Window
from src.schemas import orders_schema, restaurants_schema

spark = create_spark_session("FastFood-SQL", enable_s3=not local_data)


In [ ]:
orders_df = (
    spark.read
    .schema(orders_schema)
    .json(orders_path)
)

restaurants_df = (
    spark.read
    .schema(restaurants_schema)
    .json(restaurants_path)
)

print("Orders:", orders_df.count())
print("Restaurants:", restaurants_df.count())

orders_df.printSchema()
restaurants_df.printSchema()

orders_df.show(5, truncate=False)

In [ ]:
valid_statuses = [
    "CREATED",
    "PICKED_UP",
    "DELIVERED",
    "CANCELLED"
]

quality_report = orders_df.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.when(F.col("order_id").isNull(), 1).otherwise(0)
    ).alias("null_order_ids"),

    F.sum(
        F.when(F.col("restaurant_id").isNull(), 1).otherwise(0)
    ).alias("null_restaurant_ids"),

    F.sum(
        F.when(
            F.col("amount_mad").isNull() |
            (F.col("amount_mad") <= 0),
            1
        ).otherwise(0)
    ).alias("invalid_amounts"),

    F.sum(
        F.when(
            ~F.col("status").isin(valid_statuses) |
            F.col("status").isNull(),
            1
        ).otherwise(0)
    ).alias("invalid_statuses")
)

quality_report.show(truncate=False)

In [ ]:
report = quality_report.first().asDict()
assert report.pop("total_rows") > 0
assert all(value == 0 for value in report.values()), report

assert restaurants_df.count() == restaurants_df.select("restaurant_id").distinct().count()
assert orders_df.join(restaurants_df, "restaurant_id", "left_anti").count() == 0

events = orders_df.dropDuplicates().withColumn("event_ts", F.to_timestamp("event_time"))
assert events.filter(F.col("event_ts").isNull()).count() == 0
delivered = (events.filter(F.col("status") == "DELIVERED")
             .withColumn("amount_mad", F.col("amount_mad").cast("decimal(12,2)"))
             .dropDuplicates(["order_id"]))
delivered.createOrReplaceTempView("delivered")


In [ ]:
by_city = spark.sql("""
    SELECT city, COUNT(*) AS orders, SUM(amount_mad) AS revenue_mad,
           ROUND(AVG(amount_mad), 2) AS avg_basket_mad
    FROM delivered GROUP BY city ORDER BY city
""")
by_city.show()
assert sum(row.orders for row in by_city.collect()) == delivered.count()


from time import perf_counter

def measure(label, query):
    start = perf_counter()
    rows = query.collect()
    print(f"{label}: {perf_counter() - start:.3f} s")
    return rows

df_by_city = (delivered.groupBy("city")
              .agg(F.count("*").alias("orders"),
                   F.sum("amount_mad").alias("revenue_mad"),
                   F.round(F.avg("amount_mad"), 2).alias("avg_basket_mad"))
              .orderBy("city"))
assert measure("DataFrame par ville", df_by_city) == measure("SQL par ville", by_city)
df_by_city.show()

In [ ]:
reference = restaurants_df.select("restaurant_id", "name", "category")
join = delivered.join(reference, "restaurant_id")
join.explain(True)
(join.groupBy("city", "category")
 .agg(F.sum("amount_mad").alias("revenue_mad"))
 .orderBy("city", "category").show(30, truncate=False))

In [ ]:
@F.udf("string")
def distance_band(distance_km):
    if distance_km is None:
        return None
    if distance_km < 2:
        return "court"
    if distance_km <= 5:
        return "moyen"
    return "long"

distance = F.col("distance_km")
native_band = (F.when(distance.isNull(), F.lit(None).cast("string"))
               .when(distance < 2, "court")
               .when(distance <= 5, "moyen")
               .otherwise("long"))
sample = join.select("distance_km").cache()
sample.count()
udf_counts = sample.select(distance_band("distance_km").alias("band")).groupBy("band").count()
native_counts = sample.select(native_band.alias("band")).groupBy("band").count()
assert measure("UDF Python", udf_counts.orderBy("band")) == measure("Expression Spark", native_counts.orderBy("band"))
assert sample.filter(~distance_band("distance_km").eqNullSafe(native_band)).count() == 0
sample.unpersist()

silver = join.withColumn("distance_band", native_band)
silver_path = (Path(local_data).resolve().parent / "silver" / "orders_history").as_uri() if local_data else "s3a://lake/silver/orders_history/"
silver.write.mode("overwrite").partitionBy("city").parquet(silver_path)
reloaded = spark.read.parquet(silver_path)
assert reloaded.count() == delivered.count()
print("Silver:", silver_path)
reloaded.groupBy("city").count().orderBy("city").show()

import time
time.sleep(int(os.getenv("FASTFOOD_UI_HOLD_SECONDS", "0")))
spark.stop()